# Devon trains her own brain 🧠
**by oluwacutyp / peacethefirst1** — personal-model pipeline for the unrestrained-chat-bot.

**What this does:** takes YOUR chat data (exported from the bot), fine-tunes an open model on it with QLoRA — SFT for your voice, then DPO on your `.good`/`.bad` pairs for your taste — and pushes the result plus a phone-ready GGUF back to your HuggingFace account.

**Step 0 — turn on the free GPU:** tap **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells top to bottom (each ▶ button). Leave the tab open while training.

In [ ]:
!pip install -q transformers datasets trl peft bitsandbytes accelerate huggingface_hub

## Step 1 — log in to HuggingFace
1. Create a free token at **huggingface.co/settings/tokens** (type: **write**).
2. In Colab tap the 🔑 icon (left sidebar) → **add secret** named `HF_TOKEN` → paste the token. Future runs reuse it. (No secret? You will be asked to type it.)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except Exception:
    _tok = None
if not _tok:
    import getpass
    _tok = getpass.getpass("HF token (write access): ")
login(token=_tok)
print("logged in - token saved for this session")

## Step 2 — point at YOUR data
First, on the bot (Termux terminal or Telegram self-chat):
```
.train export
.train push USERNAME/personal-ai-pack
```
(replace `USERNAME` with your HuggingFace username — check it at huggingface.co/settings/profile.)

Then fill the 3 names below and run. The default base is small, fast, and free-T4-safe — your chats steer her voice either way. For a natively-uncensored base, swap `BASE_MODEL` to one of the commented options — but 7B+ models need a bigger GPU (Colab Pro A100) or they will OOM on a free T4.

In [ ]:
DATASET_REPO = "USERNAME/personal-ai-pack"   # <- your pushed training data
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"    # free-T4-safe default
NEW_MODEL    = "USERNAME/devon-3b"           # <- where she will live
SEQ_LEN = 1024
# Natively-uncensored base options (need bigger GPU for 7B+):
# BASE_MODEL = "Orenguteng/Llama-3.1-8B-Lexi-Uncensored"
# BASE_MODEL = "cognitivecomputations/Dolphin3.0-Mistral-24B"  # datacenter GPU only
assert "USERNAME" not in DATASET_REPO + NEW_MODEL, "edit the 3 names above first!"
print("config ok")

## Step 3 — load your data

In [ ]:
from datasets import load_dataset
sft = load_dataset("json", data_files="hf://datasets/" + DATASET_REPO + "/sft.jsonl")["train"]
print("sft rows:", len(sft))
print("sample user:", sft[0]["messages"][1]["content"][:200])
try:
    dpo = load_dataset("json", data_files="hf://datasets/" + DATASET_REPO + "/dpo.jsonl")["train"]
    print("dpo pairs:", len(dpo))
except Exception:
    dpo = None
    print("no dpo.jsonl yet (.good/.bad more chats!) - DPO step will skip.")
assert len(sft) >= 10, "need at least 10 SFT rows - chat more, then re-export!"
EPOCHS = max(1, min(5, 2000 // max(1, len(sft))))
print("sft epochs:", EPOCHS)

## Step 4 — SFT: teach her your voice (QLoRA 4-bit)
Takes ~10-40 min on a T4 depending on data size. ☕

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map="auto")
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)

def _fmt(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}
sft_ds = sft.map(_fmt, remove_columns=sft.column_names)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
sft_args = TrainingArguments(output_dir="/content/out-sft", num_train_epochs=EPOCHS,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=2e-4, warmup_ratio=0.03, logging_steps=10,
    save_strategy="no", fp16=True, bf16=False,
    gradient_checkpointing=True, report_to="none")
try:
    tr = SFTTrainer(model=model, train_dataset=sft_ds, args=sft_args,
                    peft_config=lora, processing_class=tok,
                    dataset_text_field="text")
except TypeError:
    tr = SFTTrainer(model=model, train_dataset=sft_ds, args=sft_args,
                    peft_config=lora, tokenizer=tok)
tr.train()
tr.model.push_to_hub(NEW_MODEL + "-sft", commit_message="sft on personal chats")
tok.push_to_hub(NEW_MODEL + "-sft")
print("SFT adapter pushed")

## Step 5 — DPO: teach her your taste
Uses your `.good`/`.bad` pairs. Skips automatically if you have fewer than 8 pairs — the SFT adapter is still a full win on its own.

In [ ]:
if dpo is not None and len(dpo) >= 8:
    from trl import DPOTrainer
    try:
        from trl import DPOConfig
        dpo_args = DPOConfig(output_dir="/content/out-dpo",
            num_train_epochs=max(1, min(3, 300 // max(1, len(dpo)))),
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            learning_rate=1e-6, beta=0.1, logging_steps=10,
            save_strategy="no", fp16=True, bf16=False,
            gradient_checkpointing=True, report_to="none", max_length=SEQ_LEN)
        _kw = {}
    except ImportError:
        dpo_args = TrainingArguments(output_dir="/content/out-dpo",
            num_train_epochs=max(1, min(3, 300 // max(1, len(dpo)))),
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            learning_rate=1e-6, logging_steps=10,
            save_strategy="no", fp16=True, bf16=False,
            gradient_checkpointing=True, report_to="none")
        _kw = {"beta": 0.1}
    try:
        dtr = DPOTrainer(model=tr.model, args=dpo_args, train_dataset=dpo,
                         processing_class=tok, **_kw)
    except TypeError:
        dtr = DPOTrainer(model=tr.model, args=dpo_args, train_dataset=dpo,
                         tokenizer=tok, **_kw)
    dtr.train()
    dtr.model.push_to_hub(NEW_MODEL, commit_message="dpo on owner prefs")
    print("DPO adapter pushed")
    FINAL_ADAPTER = NEW_MODEL
else:
    print("DPO skipped - SFT adapter is the final one.")
    FINAL_ADAPTER = NEW_MODEL + "-sft"

## Step 6 — merge + build the phone-ready GGUF
Merges the adapter into the base model and converts to Q4_K_M GGUF (~2 GB for 3B) — this is the file your Termux bot runs with zero internet.

In [ ]:
!pip install -q gguf
!rm -rf /content/llama.cpp && git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

In [ ]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16,
                                            device_map="auto")
merged = PeftModel.from_pretrained(base, FINAL_ADAPTER).merge_and_unload()
merged.save_pretrained("/content/merged")
tok.save_pretrained("/content/merged")
print("merged")

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/devon-q4_k_m.gguf --outtype q4_k_m

## Step 7 — push the GGUF to your account

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=_tok)
api.create_repo(NEW_MODEL + "-gguf", exist_ok=True)
api.upload_file(path_or_fileobj="/content/devon-q4_k_m.gguf",
                path_in_repo="devon-q4_k_m.gguf", repo_id=NEW_MODEL + "-gguf")
print("GGUF pushed: " + NEW_MODEL + "-gguf")

## Step 8 — point the bot at her 🩷
Back in Termux:
```
git pull --ff-only
python bot.py models --download USERNAME/devon-3b-gguf/devon-q4_k_m.gguf
```
(note the exact path it prints!) then in Telegram self-chat:
```
.model load /data/data/com.termux/files/home/.godquant/models/devon-q4_k_m.gguf
```
(use YOUR printed path — `.model list` confirms she switched.) She is now running **your weights**, fully offline-capable. And every future `.good`/`.bad` keeps improving the *next* training round. The loop is closed. 🔁